---
title: "Lab 12: Transfer learning end-to-end --- własny klasyfikator obrazów"
jupyter: python3
---

### Biblioteki Python w analizie danych

**Tomasz Rodak**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_12.ipynb)

Na wykładzie 7 trenowaliśmy klasyfikator ras zwierząt na zbiorze Oxford-IIIT
Pets: sześć linijek `fastai`, ponad dziewięćdziesiąt procent dokładności,
trzy minuty na GPU. Ale Pets to zbiór wyselekcjonowany --- etykiety czyste,
kompozycje typowe, ktoś wcześniej odsiał śmieci. Realny projekt prawie nigdy
tak nie wygląda. Zaczyna się od **pomysłu** ("chcę odróżniać te trzy rzeczy"),
nie od gotowego zbioru, a dane, które uda się zebrać, są brudne.

W tym labie przejdziemy pełny cykl: wybór kategorii → pobranie obrazów z
internetu → trening → diagnostyka → **ręczne czyszczenie danych** → ponowny
trening → działający klasyfikator. Najważniejsza część nie jest techniczna.
Jest nią iteracja **dane ↔ model**: zobaczymy, że na małym zbiorze poprawa
danych daje więcej niż jakakolwiek zmiana modelu.

`fastai` znasz już z wykładu 7 --- `DataBlock`, `vision_learner`, `lr_find`,
`fine_tune`, `ClassificationInterpretation` nie są tu tłumaczone, tylko
stosowane.

::: {.callout-tip}
## Jeśli pobieranie nie zadziała
Dane pobiera narzędzie `image_downloader`, które pod spodem korzysta z pakietu
`ddgs` --- a ten zależy od zewnętrznego API, bywającego zmienianym. Zacznij od
podglądu `--dry-run` (sekcja 2.1): jeśli nie zbiera linków, sprawdź wersję
pakietu. Gdyby pobieranie zawiodło całkowicie (brak sieci w sali), przejdź do
**sekcji 10** po zapasowy zbiór i wróć stąd do sekcji 3.
:::

## 0. Instalacje i narzędzia

Do pobierania danych użyjemy gotowego narzędzia `image_downloader` ---
osobnego pakietu uruchamianego z linii poleceń, ze strategią grzeczną wobec
serwerów. Pobierz je do środowiska Colab i zainstaluj zależności razem z
`fastai`:

In [ ]:
# Dostosuj adres / ścieżkę do miejsca, w którym trzymasz image_downloader
!git clone -q https://github.com/rodakt/image_downloader

In [ ]:
!pip install -q fastai -r image_downloader/requirements.txt

In [ ]:
import json
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from fastai.vision.all import *

Wszystkie ścieżki niżej (`data/imgrec`, `categories.json`, `model.pkl`) są
względne do bieżącego katalogu, czyli tego, do którego weszliśmy przez `%cd`.

## 1. Wybór kategorii

Najpierw decyzja, *co* chcemy rozpoznawać. Cztery kryteria:

1. **Dobra reprezentacja w internecie.** Wyszukiwarka musi mieć z czego brać.
   Nisze typu „czcionka Garamond z lat 70." nie zadziałają.
2. **Wizualnie rozróżnialne dla człowieka.** Jeśli Ty nie odróżniasz klas po
   samym obrazie (bez kontekstu), to model prawdopodobnie też nie odróżni.
3. **Realistyczna trudność transferu.** Klasy bardzo bliskie ImageNet (np.
   konkretne rasy psów) dadzą sztucznie zawyżony wynik, bo transfer będzie za
   łatwy. 
4. **Trzy kategorie.** Ciekawsza niż problem binarny z bardziej rozbudowaną
   macierzą pomyłek.

Przykłady do wyboru lub inspiracji:

- trzy gatunki ptaków drapieżnych (myszołów, jastrząb, sokół)
- trzy typy chmur (cumulus, stratus, cirrus)
- trzy regionalne potrawy (pierogi, gołąbki, bigos)
- trzy zabytkowe samochody (Fiat 126p, Syrena, Warszawa)
- trzy dzikie koty (ryś, żbik, manul)

Kategorie zapisujemy w pliku JSON w formacie, którego oczekuje
`image_downloader`: lista obiektów `{"query": ..., "folder": ...}`. `query`
idzie do wyszukiwarki; `folder` to krótka angielska nazwa bez ukośników, która
stanie się nazwą podkatalogu i **zarazem etykietą klasy** (przez
`parent_label`), więc musi być sensowna.

In [ ]:
categories = [
    {"query": "myszołów zwyczajny", "folder": "buzzard"},
    {"query": "jastrząb gołębiarz", "folder": "goshawk"},
    {"query": "sokół wędrowny",     "folder": "peregrine"},
]
Path('categories.json').write_text(
    json.dumps(categories, ensure_ascii=False, indent=2), encoding='utf-8'
)

## 2. Pobieranie obrazów

In [ ]:
DATA_DIR = Path('data/imgrec')

`image_downloader` działa dwufazowo: najpierw zbiera linki z DuckDuckGo (jedno
zapytanie na kategorię), potem pobiera obrazy, grupując je po domenie ---
**równolegle między domenami, ale sekwencyjnie i z odstępem w obrębie jednego
hosta** (wzorzec *per-host politeness*). Nazwą pliku jest hash MD5 adresu URL,
więc ponowne uruchomienie dosypuje tylko nowe obrazy, a błędne pliki (HTTP,
timeout, niepoprawny obraz) są pomijane.

### 2.1 Podgląd (dry-run)

Zanim cokolwiek pobierzemy, `--dry-run` wykonuje samą fazę 1: pokazuje, ile
linków zebrano i jak rozkładają się po domenach --- szybki sprawdzian, że
zbieranie w ogóle działa.

In [ ]:
!python -m image_downloader --config categories.json --max-results 40 --dry-run

### 2.2 Pobieranie

In [ ]:
!python -m image_downloader --config categories.json --out data/imgrec --max-results 150

Docelowo chcemy 80--150 obrazów na kategorię. Wyszukiwarka zwraca linki z
nadmiarem na błędy pobierania, więc `--max-results 150` daje typowo 100--130
użytecznych obrazów. Z uwagi na throttling w obrębie hosta pobieranie potrwa
nieco dłużej niż „na pełnej współbieżności" --- to świadomy koszt ponoszony
wobec serwerów. Komórkę można uruchamiać wielokrotnie; dzięki deduplikacji po
hashu dorzuca tylko nowe pliki.

### 2.3 Weryfikacja zbioru

In [ ]:
for cat in categories:
    folder = cat['folder']
    n = len(list((DATA_DIR/folder).glob('*')))
    print(f'{folder}: {n} obrazów')

!du -hsc data/imgrec

### 2.4 Pierwszy rzut oka na dane

Zanim cokolwiek wytrenujemy, **spójrz** na to, co pobrałeś.

In [ ]:
def show_samples(folder, n=9):
    folder = Path(folder)
    files = random.sample(list(folder.glob('*')), n)
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, f in zip(axes.flat, files):
        ax.imshow(Image.open(f).convert('RGB'))
        ax.axis('off')
    fig.suptitle(folder.name)
    plt.tight_layout()
    plt.show()

for cat in categories:
    show_samples(DATA_DIR/cat['folder'])

Spodziewane obserwacje: większość obrazów jest w porządku, kilka jest
oczywiście błędnych (dla „sokoła wędrownego" wyszuka się logo helikoptera
Sokół), niektóre dwuznaczne (rysunki, znaki rozpoznawcze, mapy zasięgów).

**Nie czyść jeszcze ręcznie.** Chcemy najpierw zobaczyć, czego model będzie się
„czepiał" --- *top losses* w sekcji 6 powiedzą nam, które obrazy są naprawdę
problematyczne. Bywa, że to nie te, które na pierwszy rzut oka wyglądają źle.

## 3. `DataBlock` i `DataLoaders`

To samo co na wykładzie 7, zastosowane do własnego zbioru.

In [ ]:
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(224),
    batch_tfms=[*aug_transforms(), Normalize.from_stats(*imagenet_stats)],
)
dls = dblock.dataloaders(DATA_DIR, bs=32)
dls.show_batch(max_n=9)
print(dls.vocab)

Trzy rzeczy warte odnotowania:

- `get_y=parent_label` --- etykietą jest nazwa folderu, dlatego nazwy z sekcji
  1 musiały być sensowne. Sprawdź, że `dls.vocab` wypisuje Twoje trzy klasy.
- `bs=32` zamiast `64` jak na wykładzie --- mamy mniej danych (300--400
  obrazów), mniejszy batch ma sens.
- `RandomSplitter(seed=42)` --- ten sam podział przed i po czyszczeniu, więc
  porównanie będzie uczciwe.

## 4. Pierwszy trening (baseline)

In [ ]:
learn = vision_learner(dls, resnet18, metrics=accuracy)
res = learn.lr_find()

Z krzywej `lr_find` odczytujemy propozycję `res.valley` i podajemy ją wprost
do `fine_tune`:

In [ ]:
learn.fine_tune(3, base_lr=res.valley)

Dla trzech klas i ~100 obrazów na klasę ResNet-18 z transferem powinien dać
80--95% dokładności walidacyjnej w trzy epoki. Wynik poniżej ~70% to sygnał, że
coś jest nie tak (pomieszane etykiety, klasy nierozróżnialne, za mało danych).

Zapisujemy wynik bazowy, by porównać go w sekcji 8:

In [ ]:
baseline_acc = learn.recorder.values[-1][-1]   # ostatnia metryka (accuracy)
print(f'Baseline val accuracy: {baseline_acc:.4f}')

## 5. Macierz pomyłek

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(6, 6))

Odpowiedz sobie (jako komentarz w notatniku):

- Która klasa jest dla modelu najtrudniejsza?
- Czy któreś dwie klasy mylą się **w obie strony** (konfuzja symetryczna), czy
  tylko w jedną?
- Czy to ma sens wobec ich wizualnego podobieństwa?

## 6. Top losses --- diagnostyka jakościowa

In [ ]:
interp.plot_top_losses(12, nrows=3, figsize=(16, 12))

Dla każdego obrazu widzisz etykietę prawdziwą, predykcję, pewność i wartość
straty. **Sklasyfikuj każdy z tych obrazów** do jednej z trzech kategorii (w
komórce markdown poniżej):

1. **Błędna etykieta** --- obraz nie przedstawia naszej klasy (zły obiekt,
   logo, schemat, rysunek), choć leży w jej folderze. **Do usunięcia.**
2. **Trudny, ale prawdziwy** --- faktycznie nasza klasa, tylko trudna (zła
   kompozycja, stadium młodociane, częściowe ujęcie). **Zostaje** --- to
   legalne dane treningowe.
3. **Dwuznaczny** --- mógłby należeć do innej z naszych klas (np. ptak z
   bardzo daleka, nie do odróżnienia).

Cel: nauczyć się odróżniać **błąd danych** (1) od **naturalnej trudności** (2).
Pierwsze usuwamy, drugie zostawiamy --- model ma się nauczyć radzić sobie z
trudnymi przypadkami, nie z błędami w etykietach.

## 7. Ręczne czyszczenie zbioru

::: {.callout-important}
## Czy patrzenie na zbiór walidacyjny to oszustwo?
Na lab 5 i 6 byliśmy rygorystyczni co do *data leakage*. Tutaj patrzymy na
straty na zbiorze walidacyjnym i na tej podstawie modyfikujemy dane --- czy to
nie ta sama pułapka? **Nie**, pod warunkiem, że robimy to dobrze.

Leakage to wstrzyknięcie informacji ze zbioru walidacyjnego do **treningu**.
My robimy coś innego: usuwamy **szum etykiet**. Obraz oznaczony „buzzard",
który w rzeczywistości jest logiem, niesłusznie karał model za poprawną
decyzję --- jego usunięcie czyni wynik *prawdziwszym*, a nie zawyżonym.
Większość zysku i tak pochodzi z czystszego **sygnału treningowego**.

Granica jest jedna i ostra: usuwamy wyłącznie obrazy z kategorii 1 (błędna
etykieta). Gdybyśmy usuwali obrazy *trudne, ale poprawne* (kategoria 2) tylko
dlatego, że model się na nich myli --- to byłoby cherry-picking i zawyżenie
wyniku. Tego nie wolno robić.

Dlatego porównanie z sekcji 8 traktujemy jako **diagnostykę efektu
czyszczenia**, a nie jako ostateczny, certyfikowany benchmark. Po
zmodyfikowaniu zbioru walidacyjnego ścisła ocena wymagałaby osobnego, nigdy
nieruszanego zbioru testowego. To jest standardowa praktyka inżynierska ---
dokładnie to, co automatyzuje narzędzie `ImageClassifierCleaner` z fastai.
:::

### 7.1 Identyfikacja obrazów do usunięcia

Z `top_losses(items=True)` dostajemy posortowane straty, indeksy oraz gotowe
ścieżki plików:

In [ ]:
losses, idxs, paths = interp.top_losses(k=30, items=True)
for rank, (loss, idx, path) in enumerate(zip(losses, idxs, paths)):
    true = dls.vocab[int(interp.targs[idx])]
    pred = dls.vocab[int(interp.decoded[idx])]
    print(f'{rank:2d}. loss={loss:.2f} | true={true:>10} | pred={pred:>10} | {path}')

Przejrzyj wypis i wpisz ścieżki obrazów z kategorii „błędna etykieta" do listy
`to_remove`:

In [ ]:
to_remove = [
    # 'data/imgrec/peregrine/abc123.jpeg',  # logo helikoptera
    # 'data/imgrec/buzzard/def456.png',     # schemat zasięgu
]

### 7.2 Usunięcie plików i weryfikacja

In [ ]:
for path in to_remove:
    Path(path).unlink()
print(f'Usunięto {len(to_remove)} obrazów.\n')

for cat in categories:
    folder = cat['folder']
    n = len(list((DATA_DIR/folder).glob('*')))
    print(f'{folder}: {n} obrazów')

## 8. Trening na oczyszczonym zbiorze

Powtarzamy sekcje 3--4 od czystego stanu --- nowy `DataLoaders`, nowy
`Learner`, nowe `lr_find` i `fine_tune`. **Ten sam seed** (`seed=42`), więc
porównujemy wynik bazowy i po czyszczeniu na tym samym, recomputowanym dla
oczyszczonego zbioru, podziale 80/20.

In [ ]:
dls_clean = dblock.dataloaders(DATA_DIR, bs=32)
learn_clean = vision_learner(dls_clean, resnet18, metrics=accuracy)
res = learn_clean.lr_find()
learn_clean.fine_tune(3, base_lr=res.valley)

clean_acc = learn_clean.recorder.values[-1][-1]
print(f'\nBaseline:    {baseline_acc:.4f}')
print(f'Po czyszczeniu: {clean_acc:.4f}')
print(f'Poprawa: {(clean_acc - baseline_acc) * 100:+.2f} pp')

Spodziewana poprawa to 2--8 punktów procentowych, zależnie od tego, ile
faktycznie złych obrazów było w zbiorze. Ten zysk pochodzi z usunięcia kilkunastu obrazów, nie ze zmiany
architektury, hiperparametrów ani z dłuższego treningu. Na małych zbiorach ---
i w większości realnych projektów ML --- poprawianie danych daje więcej niż
poprawianie modelu.

In [ ]:
interp_clean = ClassificationInterpretation.from_learner(learn_clean)
interp_clean.plot_confusion_matrix(figsize=(6, 6))

Porównaj z sekcją 5: które klasy poprawiły się najbardziej?

## 9. Predykcja na nowych obrazach

Sprawdźmy, że model działa na zdjęciach spoza zbioru. Wybierz 2--3 adresy URL
(np. z Wikimedia Commons), pobierz je `download_url` z fastai.

In [ ]:
test_urls = [
    # 'https://upload.wikimedia.org/...',
]
test_paths = []
for i, url in enumerate(test_urls):
    p = Path(f'test_{i}.jpg')
    download_url(url, p, show_progress=False)
    test_paths.append(p)

In [ ]:
for p in test_paths:
    pred, pred_idx, probs = learn_clean.predict(p)
    print(f'{p.name}: {pred} (pewność: {probs[pred_idx]:.1%})')
    display(Image.open(p).resize((224, 224)))

### 9.1 Eksport modelu

In [ ]:
learn_clean.export('model.pkl')

Plik `model.pkl` zawiera architekturę, wagi i cały pipeline transformacji ---
wczytasz go w dowolnym środowisku przez `load_learner('model.pkl')`. Do
udostępnienia modelu jako aplikacji służą narzędzia takie jak Gradio,
Streamlit czy FastAPI, ale to już wykracza poza zakres kursu.

## 10. Zbiór zapasowy (gdy pobieranie zawiedzie)

Jeśli `image_downloader` nie zadziała, użyj fragmentu zbioru Pets z wykładu 7.
Ma on 37 klas; ograniczamy się do trzech wybranych ras (etykieta jest w nazwie
pliku, np. `Bengal_101.jpg`).

In [ ]:
path = untar_data(URLs.PETS)
WANTED = ['Bengal', 'beagle', 'Sphynx']   # dwa koty (z wielkiej litery) i pies

files = [f for f in get_image_files(path/'images')
         if re.match(r'(.+)_\d+\.jpg$', f.name).group(1) in WANTED]

dblock_pets = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=lambda _: files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=using_attr(RegexLabeller(r'^(.+)_\d+.jpg$'), 'name'),
    item_tfms=Resize(224),
    batch_tfms=[*aug_transforms(), Normalize.from_stats(*imagenet_stats)],
)
dls = dblock_pets.dataloaders(path/'images', bs=32)
print(dls.vocab)

Mając ten `dls`, kontynuuj od sekcji 4 (trening, interpretacja, top losses).
Czyszczenie z sekcji 7 ma tu mniejszy sens --- Pets jest wstępnie oczyszczony
--- ale kilka błędnych etykiet i tak znajdziesz.